# Morphological Patient Similarity Retrieval

## What This Notebook Does

Given a query patient from the HANCOCK cohort, this notebook finds the most morphologically and clinically similar patients — ranked by a composite score that combines three signals:

| Signal | Weight | What it captures |
|---|---|---|
| Slide-level cosine similarity | α = 0.4 | Overall morphological proximity in 1536-d ABMIL embedding space |
| Composition profile similarity | β = 0.4 | Patch-level tissue cluster proportions — heterogeneity that mean-pooling discards |
| Clinical metadata match | γ = 0.2 | Exact-match fraction across tumor site, histology, HPV, pT/pN stage |

The output is a ranked patient table suitable for clinical trial matching. UMAP is used for visual exploration only — all matching is done in the original high-dimensional space.

**Prerequisites:** run `01_Metadata_Builder.ipynb` first to produce `enriched_slide_embeddings.csv`.

---

Given a query slide from the HANCOCK cohort, find morphologically similar slides and rank candidate patients for clinical trial matching. The pipeline combines:
- **Cosine KNN** in the 1536-d H-optimus-0 embedding space for slide-level morphological proximity
- **Tissue composition profiles** (patch-level morphological cluster proportions from H5 files) for structural similarity
- **Clinical metadata matching** across staging, HPV status, histologic type, and other filterable fields

UMAP is used as a secondary visual exploration tool only — all matching is done in the original high-dimensional space.


---

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../common").resolve()))

import numpy as np

import os
import subprocess
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize
import re

from umap_retrieval import *

np.random.seed(SEED)


from dotenv import load_dotenv
load_dotenv()
os.environ['AWS_PROFILE'] = 'pathologyworkshop'
os.environ['AWS_DEFAULT_REGION'] = 'us-west-2'

DATA_DIR = Path("../data")

## Load & Validate CSV Metadata

Loads `enriched_slide_embeddings.csv` and validates structural integrity before any downstream analysis.

### CSV structure

Each row represents one whole-slide image (WSI). Columns are grouped as:

| Group | Columns |
|---|---|
| Identity | `slide_name` |
| Filterable clinical metadata (10) | `primary_tumor_site`, `pT_stage`, `pN_stage`, `grading_hpv`, `hpv_association_p16`, `histologic_type`, `survival_status`, `recurrence`, `smoking_status`, `sex` |
| Non-filterable clinical metadata (5) | `perineural_invasion_Pn`, `lymphovascular_invasion`, `perinodal_invasion`, `age_at_initial_diagnosis`, `primarily_metastasis` |
| H5 pointer | `h5file` — S3 URI to the patch-level embedding file for this slide |
| Embedding | `embedding_dim`, then `e_0` … `e_1535` — the 1536-d H-optimus-0 slide embedding |

Slides with all-zero embeddings are dropped after loading (failed quality check at embedding time).


In [14]:
# load-validate-csv

csv_df = load_and_validate_csv(DATA_DIR/"enriched_slide_embeddings.csv")
print(f"Validated {len(csv_df)} slides with {len(csv_df.columns)} columns")
csv_df.head()

# Drop rows where embedding values are all zeros (slides that did not meet quality requirements)
embedding_cols = [col for col in csv_df.columns if col.startswith('e_')]
dropped_zero_csv_df = csv_df.loc[~(csv_df[embedding_cols] == 0).all(axis=1)]
dropped_zero_csv_df.shape
csv_df = dropped_zero_csv_df.copy()

metadata_df, embeddings = extract_metadata_and_embeddings(csv_df)
umap_df = run_umap(embeddings)

print(f"Loaded {len(metadata_df)} slides with {len(metadata_df.columns)} metadata columns")
print(f"Extracted {len(embeddings)} embeddings of dimension {len(next(iter(embeddings.values())))}")
metadata_df.head()


Validated 1534 slides with 1555 columns


/Users/ganttmeredith/Desktop/Kiro_BioFM/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Loaded 1442 slides with 18 metadata columns
Extracted 1442 embeddings of dimension 1536


,slide_name,primary_tumor_site,pT_stage,pN_stage,grading_hpv,hpv_association_p16,histologic_type,year_of_initial_diagnosis,survival_status,recurrence,smoking_status,sex,perineural_invasion_Pn,lymphovascular_invasion,perinodal_invasion,age_at_initial_diagnosis,primarily_metastasis,h5file
0,TumorCenter_HE_block10_x1_y10_patient492,Oropharynx,pT2,pN2a,G3,negative,SCC_Sarcomatoid,2012,living,no,non-smoker,male,no,no,no,57,no,../data/h5_cache/TumorCenter_HE_block10_x1_y10...
2,TumorCenter_HE_block10_x1_y12_patient411,Oral_Cavity,pT1,pN0,G2,not_tested,SCC_Conventional-Keratinizing,2014,living,no,former,male,no,no,unknown,59,no,../data/h5_cache/TumorCenter_HE_block10_x1_y12...
3,TumorCenter_HE_block10_x1_y1_patient246,Oropharynx,pT2,pN2b,G3,negative,SCC_Basaloid,2013,living,no,smoker,male,no,no,no,40,no,../data/h5_cache/TumorCenter_HE_block10_x1_y1_...
4,TumorCenter_HE_block10_x1_y2_patient222,Oropharynx,pT1,pN2b,G3,negative,SCC_Conventional-Keratinizing,2010,living,no,non-smoker,male,no,yes,no,78,no,../data/h5_cache/TumorCenter_HE_block10_x1_y2_...
5,TumorCenter_HE_block10_x1_y4_patient432,Oropharynx,pT2,pN0,G3,negative,SCC_Basaloid,2013,living,yes,smoker,male,no,no,unknown,72,no,../data/h5_cache/TumorCenter_HE_block10_x1_y4_...


## Embedding Quality Diagnostics

In this pipeline, H-optimus-0 embeddings are used at the **slide level**: patch embeddings are aggregated into a single 1536-d vector via gated attention-based MIL (ABMIL), stored in `enriched_slide_embeddings.csv`. This slide-level vector is what KNN retrieval and the composite match table operate on.

The key question this cell addresses: **how much clinically interpretable signal survives the aggregation step?** A model that produces excellent patch-level features may lose discriminative power when aggregated across thousands of patches from a heterogeneous slide — particularly when the aggregator weights are not task-trained.

### What this cell measures

Before running retrieval or building the match table, this cell answers two questions:

1. **Do the embeddings encode clinically meaningful structure?** — silhouette scores measure how well the 1536-d embedding space separates known clinical labels. A score near 0 means the embeddings don't cluster by that label; negative means worse than random.
2. **What does a "high" similarity score actually mean?** — the null distribution establishes a population baseline. A KNN neighbor at cosine similarity 0.75 is only meaningful if the typical random pair scores much lower.

Both outputs are prerequisites for interpreting the match table in `match-table`:
- Silhouette scores set expectations — if `histologic_type` has a silhouette of −0.08, don't expect KNN to reliably return same-histology matches
- `null_dist["mean"]` and `null_dist["p95"]` are used directly in `match-table` to compute `slide_sim_vs_baseline`


In [15]:
# embedding-quality-diagnostics

LABEL_COLS = [
    "primary_tumor_site",
    "hpv_association_p16",
    "histologic_type",
    "smoking_status",
    "pT_stage",
]

from IPython.display import display, HTML

# --- Silhouette scores ---
sil_df = embedding_silhouette_scores(embeddings, metadata_df, LABEL_COLS)
display(HTML("<h4 style='margin-bottom:6px'>Silhouette Scores — 1536-d cosine, patients deduplicated</h4>"))
display(
    sil_df.style
    .background_gradient(subset=["silhouette_score"], cmap="RdYlGn", vmin=-0.15, vmax=0.15)
    .format({"silhouette_score": "{:.4f}"})
    .hide(axis="index")
)

# --- Null distribution ---
null_dist = cosine_similarity_null_distribution(embeddings, sample_n=300)
null_df = pd.DataFrame([
    {"statistic": k, "value": v} for k, v in null_dist.items()
])
display(HTML("<h4 style='margin-top:18px;margin-bottom:6px'>Pairwise Cosine Similarity — Null Distribution (n=300 sample)</h4>"))
display(null_df.style.hide(axis="index"))

# --- Interpretation banner ---
display(HTML(f"""
    <div style='background:#e8f4f8;border-left:5px solid #2196F3;padding:10px 16px;
                border-radius:4px;font-family:monospace;margin-top:12px'>
    <b>Interpretation</b><br>
    Mean pairwise similarity = <b>{null_dist['mean']:.3f} ± {null_dist['std']:.3f}</b><br>
    95th percentile = <b>{null_dist['p95']:.3f}</b><br>
    A KNN neighbor is only meaningfully similar if its score is
    well above the p95 baseline (<b>{null_dist['p95']:.3f}</b>).
    </div>
"""))



label,silhouette_score,n_slides,n_classes,min_class_size
hpv_association_p16,0.0153,720,3,131
primary_tumor_site,-0.0127,720,4,82
smoking_status,-0.0529,720,4,15
histologic_type,-0.0804,718,7,4
pT_stage,-0.0835,719,8,2


statistic,value
mean,0.436700
std,0.133000
p5,0.219800
p25,0.339700
median,0.435300
p75,0.531700
p95,0.658500
n_pairs,44850.000000


### Interpreting the results

#### What the numbers say

**Silhouette scores:**

| Label | Score | Interpretation |
|---|---|---|
| `hpv_association_p16` | +0.015 | Weakest positive signal — HPV status leaves a faint trace in morphology |
| `primary_tumor_site` | −0.013 | No separation — oropharynx, larynx, oral cavity are morphologically indistinguishable at slide level |
| `smoking_status` | −0.054 | Poorly separated — same smoking-status patients are on average closer to a different group's centroid than their own |
| `histologic_type` | −0.082 | Poorly separated — same histologic type patients are on average closer to a different histology's centroid than their own |
| `pT_stage` | −0.083 | Poorly separated — same pT stage patients are on average closer to a different stage's centroid than their own |

H-optimus-0 was pretrained on general morphological patterns across many cancer types. HNSCC subtypes are all squamous carcinomas with heavily overlapping H&E appearance. ABMIL aggregation with randomly initialised weights further dilutes any subtype-specific signal — the aggregator produces a random weighted average of patch embeddings rather than a uniform mean, and without task-specific training the attention scores reflect random projections rather than learned diagnostic relevance.

**Null distribution — narrow signal window:**
- Population mean similarity = 0.437, p95 = 0.659 — only a 0.22 spread between an average pair and the top 5%
- Any match table result with `slide_sim` ≤ 0.66 is within normal population noise and should be treated with caution
- Results with `slide_sim_vs_baseline` clearly positive (> 0.05) are the ones worth examining



### Improving signal: training the ABMIL aggregator and fine-tuning H-optimus-0

Two independent levers can improve slide-level embedding quality. The first is faster and more impactful for this cohort size; the second is more expensive but improves the underlying patch features.

#### Option 1 - Fine-tune H-optimus-0 with LoRA

H-optimus-0 was pretrained with self-supervised objectives across a broad pan-cancer corpus — it learned general morphological texture, not HNSCC-specific subtype discrimination. Domain adaptation via LoRA fine-tuning directly optimises the patch-level embedding space for the labels that matter here.

A LoRA adapter (rank 16–32) on the final 4–6 transformer blocks is sufficient — no need to update the full 1.1B parameters.

- Expected outcome: `histologic_type` and `primary_tumor_site` silhouette scores move from −0.08 to +0.10–0.20 range
- Training data: the existing 763-patient HANCOCK cohort (~720 usable after zero-embedding filtering) with leave-one-out cross-validation to avoid label leakage
- Loss: `SupConLoss` from `pytorch-metric-learning`, or a triplet margin loss
- Compute: requires GPU — best suited for SageMaker HyperPod multi-node runs

#### Option 2 — Self-supervised continued pretraining on HNSCC patches

Run masked image modelling (MIM) or DINO-style self-distillation on the full Hancock cohort patch library before any label-supervised step. This adapts patch-level representations to HNSCC tissue morphology without requiring labels.

- Expected outcome: improved baseline similarity distribution (higher mean, tighter std), better patch-level composition profiles
- No labels required — uses the H5 patch files already available via `h5file` column
- Higher compute cost than Options 1 and 2 — best suited for HyperPod multi-node runs

#### Running GPU training on Amazon SageMaker HyperPod

[Amazon SageMaker HyperPod](https://aws.amazon.com/sagemaker/hyperpod/) is purpose-built for large-scale distributed training. For Options 2 and 3, [HyperPod training recipes](https://github.com/aws/sagemaker-hyperpod-training-adapter-for-nemo) provide ready-made ViT + LoRA configs, reducing setup to parameter overrides. Option 1 (ABMIL aggregator training) does not require HyperPod — a single `ml.g4dn.xlarge` SageMaker Training job is sufficient.



---

## Unsupervised Clustering in 1536-d Embedding Space

Sweeps K=2..10, clustering patients by their ABMIL-aggregated 1536-d embeddings (one vector per patient). The best K's labels are cross-tabulated against clinical metadata to reveal what structure the embeddings actually organise by.

Silhouette scores here are computed in the original embedding space — not on UMAP coordinates — so they are consistent with the diagnostics in `embedding-quality-diagnostics`. Given the near-zero scores seen there, expect weak cluster structure. Use the cross-tabulation heatmap to identify any dominant clinical signal within the best-K clusters, then apply that as a metadata filter in the cells below.

> **Algorithm note**: `cluster_embeddings` uses exact `KMeans` on L2-normalised embeddings (not `MiniBatchKMeans`). This is valid for cosine-oriented clustering because on the unit sphere, minimizing Euclidean distance is equivalent to maximizing cosine similarity — `‖u−v‖² = 2 − 2·cos(u,v)` when `‖u‖=‖v‖=1` — so standard KMeans centroids optimise the same objective as spherical KMeans. At ~720 patients the matrix is ~4 MB — exact KMeans is fast and gives fully deterministic results with a fixed seed. `MiniBatchKMeans` is used in `composition-k-sweep` because that sweep operates on hundreds of thousands of patch vectors where the mini-batch approximation is the practical choice. The two algorithms are not interchangeable here: using `MiniBatchKMeans` on 720 points would trade reproducibility for no meaningful speed gain.


In [16]:
# unsupervised-clustering

silhouette_df, cluster_df = cluster_embeddings(
    embeddings,
    metadata_df,
    k_range=range(2, 11),
    cross_tabulate_cols=["primary_tumor_site", "histologic_type", "hpv_association_p16"],
    verbose=False
)

plot_clustering_results(
    silhouette_df,
    cluster_df,
    cross_tabulate_cols=["primary_tumor_site", "histologic_type", "hpv_association_p16"],
).show()



## Metadata Filtering — Selected vs Excluded Slides

### Goal

Before running KNN retrieval or building the match table, you typically want to restrict the candidate pool to a clinically relevant subgroup. These two cells demonstrate filtering at different levels of specificity — a single-field filter (`DEMO_FILTERS`) and a tighter two-field filter (`MULTI_FILTERS`) — so you can see how adding constraints progressively narrows the pool.

### What you learn

- How `apply_metadata_filters` works: AND logic across fields, OR logic within each field's value list
- The size impact of each filter — important for knowing whether downstream KNN has enough candidates (too few → reduce k)
- Which slides are explicitly excluded, so you can audit the split or export either group

### How filters connect to downstream cells

`MULTI_FILTERS` is reused in `multi-filter-umap-demo` to restrict the UMAP view, and can be passed as `metadata_filters` to any KNN call to restrict the candidate pool. Filtering before KNN is the primary way to compensate for weak embedding silhouette scores — if the embeddings don't reliably separate a label, explicitly restricting candidates enforces that constraint directly.

> All 10 filterable columns are valid filter keys: `primary_tumor_site`, `pT_stage`, `pN_stage`, `grading_hpv`, `hpv_association_p16`, `histologic_type`, `survival_status`, `recurrence`, `smoking_status`, `sex`. See the Glossary for definitions.


In [17]:
# metadata-filter-demo

DEMO_FILTERS = {"primary_tumor_site": ["Oral_Cavity"]}

selected_df = apply_metadata_filters(metadata_df, DEMO_FILTERS, FILTERABLE_COLUMNS)
excluded_df = metadata_df[~metadata_df["slide_name"].isin(selected_df["slide_name"])].copy()

total = len(metadata_df)
n_selected = len(selected_df)
n_excluded = len(excluded_df)

print(f"Filter criteria: {DEMO_FILTERS}")
print(f"Total slides in dataset: {total}")
print(f"Selected (passed all filters): {n_selected} ({n_selected/total:.1%})")
print(f"Excluded (did not match): {n_excluded} ({n_excluded/total:.1%})")
print()
print("--- Selected slides (first 10) ---")
display(selected_df.head(10))



Filter criteria: {'primary_tumor_site': ['Oral_Cavity']}
Total slides in dataset: 1442
Selected (passed all filters): 260 (18.0%)
Excluded (did not match): 1182 (82.0%)

--- Selected slides (first 10) ---


,slide_name,primary_tumor_site,pT_stage,pN_stage,grading_hpv,hpv_association_p16,histologic_type,year_of_initial_diagnosis,survival_status,recurrence,smoking_status,sex,perineural_invasion_Pn,lymphovascular_invasion,perinodal_invasion,age_at_initial_diagnosis,primarily_metastasis,h5file
2,TumorCenter_HE_block10_x1_y12_patient411,Oral_Cavity,pT1,pN0,G2,not_tested,SCC_Conventional-Keratinizing,2014,living,no,former,male,no,no,unknown,59,no,../data/h5_cache/TumorCenter_HE_block10_x1_y12...
13,TumorCenter_HE_block10_x2_y12_patient411,Oral_Cavity,pT1,pN0,G2,not_tested,SCC_Conventional-Keratinizing,2014,living,no,former,male,no,no,unknown,59,no,../data/h5_cache/TumorCenter_HE_block10_x2_y12...
75,TumorCenter_HE_block11_x1_y4_patient065,Oral_Cavity,pT2,NX,G3,not_tested,SCC_Conventional-Keratinizing,2011,living,yes,smoker,male,yes,yes,unknown,59,unknown,../data/h5_cache/TumorCenter_HE_block11_x1_y4_...
86,TumorCenter_HE_block11_x2_y4_patient065,Oral_Cavity,pT2,NX,G3,not_tested,SCC_Conventional-Keratinizing,2011,living,yes,smoker,male,yes,yes,unknown,59,unknown,../data/h5_cache/TumorCenter_HE_block11_x2_y4_...
93,TumorCenter_HE_block11_x3_y11_patient733,Oral_Cavity,pT1,pN0,G2,not_tested,SCC_Conventional-Keratinizing,2006,living,no,unknown,male,no,no,unknown,54,no,../data/h5_cache/TumorCenter_HE_block11_x3_y11...
105,TumorCenter_HE_block11_x4_y11_patient733,Oral_Cavity,pT1,pN0,G2,not_tested,SCC_Conventional-Keratinizing,2006,living,no,unknown,male,no,no,unknown,54,no,../data/h5_cache/TumorCenter_HE_block11_x4_y11...
117,TumorCenter_HE_block11_x5_y11_patient708,Oral_Cavity,pT2,pN0,G2,not_tested,SCC_Conventional-Keratinizing,2012,deceased,no,smoker,male,no,no,unknown,71,no,../data/h5_cache/TumorCenter_HE_block11_x5_y11...
127,TumorCenter_HE_block11_x5_y9_patient509,Oral_Cavity,pT4a,pN2b,G2,not_tested,SCC_Conventional-Keratinizing,2012,living,no,smoker,male,yes,no,yes,61,no,../data/h5_cache/TumorCenter_HE_block11_x5_y9_...
129,TumorCenter_HE_block11_x6_y11_patient708,Oral_Cavity,pT2,pN0,G2,not_tested,SCC_Conventional-Keratinizing,2012,deceased,no,smoker,male,no,no,unknown,71,no,../data/h5_cache/TumorCenter_HE_block11_x6_y11...
139,TumorCenter_HE_block11_x6_y9_patient509,Oral_Cavity,pT4a,pN2b,G2,not_tested,SCC_Conventional-Keratinizing,2012,living,no,smoker,male,yes,no,yes,61,no,../data/h5_cache/TumorCenter_HE_block11_x6_y9_...


### Multi-criteria filter — oropharynx, HPV-positive only

Tightens the filter to `Oropharynx` AND `hpv_association_p16 = positive`. This targets the HPV-driven subgroup of oropharyngeal SCC — morphologically and clinically distinct from HPV-negative cases.

The AND logic across two fields is what makes this a multi-filter:
- `primary_tumor_site` must be `Oropharynx`
- **and** `hpv_association_p16` must be `positive`

`MULTI_FILTERS` is passed to `multi-filter-umap-demo` (colored by `hpv_association_p16`) and can be reused in any downstream KNN call. Modify the values to target a different subgroup.


In [18]:
# multi-filter-demo

MULTI_FILTERS = {
    "primary_tumor_site": ["Oropharynx"],
    "hpv_association_p16": ["positive", "negative"],
}

selected_multi_df = apply_metadata_filters(metadata_df, MULTI_FILTERS, FILTERABLE_COLUMNS)
excluded_multi_df = metadata_df[
    ~metadata_df["slide_name"].isin(selected_multi_df["slide_name"])
].copy()

print(f"Filter criteria: {MULTI_FILTERS}")
print(f"Selected: {len(selected_multi_df)} / {len(metadata_df)} "
      f"({len(selected_multi_df)/len(metadata_df):.1%})")
print(f"Excluded: {len(excluded_multi_df)} / {len(metadata_df)} "
      f"({len(excluded_multi_df)/len(metadata_df):.1%})")
print()
print("--- Selected slides (first 10) ---")
display(selected_multi_df.head(10))



Filter criteria: {'primary_tumor_site': ['Oropharynx'], 'hpv_association_p16': ['positive', 'negative']}
Selected: 626 / 1442 (43.4%)
Excluded: 816 / 1442 (56.6%)

--- Selected slides (first 10) ---


,slide_name,primary_tumor_site,pT_stage,pN_stage,grading_hpv,hpv_association_p16,histologic_type,year_of_initial_diagnosis,survival_status,recurrence,smoking_status,sex,perineural_invasion_Pn,lymphovascular_invasion,perinodal_invasion,age_at_initial_diagnosis,primarily_metastasis,h5file
0,TumorCenter_HE_block10_x1_y10_patient492,Oropharynx,pT2,pN2a,G3,negative,SCC_Sarcomatoid,2012,living,no,non-smoker,male,no,no,no,57,no,../data/h5_cache/TumorCenter_HE_block10_x1_y10...
3,TumorCenter_HE_block10_x1_y1_patient246,Oropharynx,pT2,pN2b,G3,negative,SCC_Basaloid,2013,living,no,smoker,male,no,no,no,40,no,../data/h5_cache/TumorCenter_HE_block10_x1_y1_...
4,TumorCenter_HE_block10_x1_y2_patient222,Oropharynx,pT1,pN2b,G3,negative,SCC_Conventional-Keratinizing,2010,living,no,non-smoker,male,no,yes,no,78,no,../data/h5_cache/TumorCenter_HE_block10_x1_y2_...
5,TumorCenter_HE_block10_x1_y4_patient432,Oropharynx,pT2,pN0,G3,negative,SCC_Basaloid,2013,living,yes,smoker,male,no,no,unknown,72,no,../data/h5_cache/TumorCenter_HE_block10_x1_y4_...
6,TumorCenter_HE_block10_x1_y5_patient443,Oropharynx,pT2,pN1,hpv_association_p16,positive,SCC_Conventional-Keratinizing,2013,living,no,non-smoker,male,no,no,no,62,no,../data/h5_cache/TumorCenter_HE_block10_x1_y5_...
8,TumorCenter_HE_block10_x1_y7_patient038,Oropharynx,pT2,pN2,hpv_association_p16,positive,SCC_Basaloid,2013,living,no,smoker,male,no,yes,no,55,no,../data/h5_cache/TumorCenter_HE_block10_x1_y7_...
10,TumorCenter_HE_block10_x1_y9_patient427,Oropharynx,pT2,pN2b,hpv_association_p16,positive,SCC_Conventional-Keratinizing,2013,living,no,former,male,no,no,no,54,no,../data/h5_cache/TumorCenter_HE_block10_x1_y9_...
11,TumorCenter_HE_block10_x2_y10_patient492,Oropharynx,pT2,pN2a,G3,negative,SCC_Sarcomatoid,2012,living,no,non-smoker,male,no,no,no,57,no,../data/h5_cache/TumorCenter_HE_block10_x2_y10...
14,TumorCenter_HE_block10_x2_y1_patient246,Oropharynx,pT2,pN2b,G3,negative,SCC_Basaloid,2013,living,no,smoker,male,no,no,no,40,no,../data/h5_cache/TumorCenter_HE_block10_x2_y1_...
15,TumorCenter_HE_block10_x2_y2_patient222,Oropharynx,pT1,pN2b,G3,negative,SCC_Conventional-Keratinizing,2010,living,no,non-smoker,male,no,yes,no,78,no,../data/h5_cache/TumorCenter_HE_block10_x2_y2_...


### Filter comparison

The two filter cells above show how tightening criteria progressively narrows the candidate pool:

| Filter | Criteria | Selected | % of dataset |
|---|---|---|---|
| `DEMO_FILTERS` | `primary_tumor_site = Oral_Cavity` | 260 slides | 18.0% |
| `MULTI_FILTERS` | `primary_tumor_site = Oropharynx` **AND** `hpv_association_p16 = positive or negative` | 626 slides | 43.4% |

Note: Pool size alone doesn't determine filter quality. `DEMO_FILTERS` is a broad single-site filter; `MULTI_FILTERS` is a two-field intersection targeting a clinically specific subgroup. `MULTI_FILTERS` is what flows into the UMAP and KNN cells below.

`MULTI_FILTERS` is carried forward into the UMAP plot and KNN cells below.


## UMAP Visualization

### Filtered UMAP — HPV-positive oropharynx, colored by HPV status

Applies `MULTI_FILTERS` (Oropharynx + HPV) and colors by `hpv_association_p16`. With 626 slides from the HPV-positive oropharyngeal subgroup, this is the most clinically coherent view the current embeddings can produce.

Compare this visually to the full-dataset UMAP in `umap-computation` — the filtered view should show a denser, more localised cluster. Any residual mixing with HPV-negative points visible here reflects the weak silhouette (+0.015) and is expected.

Try changing `color_by` to `pT_stage`, `smoking_status`, or `histologic_type` to confirm those axes show no structure — consistent with their negative silhouette scores.


In [19]:
# multi-filter-umap-demo

fig_filtered = plot_umap_interactive(
    umap_df, metadata_df,
    color_by="hpv_association_p16",
    metadata_filters=MULTI_FILTERS,
)
fig_filtered.show()


626 slides remaining after filtering


### Query by Smoking Status

Demonstrates slide-level KNN retrieval with both a query selector and a candidate pool filter applied together. `MULTI_FILTERS` (Oropharynx + HPV positive or negative, 626 slides) restricts the candidate pool. Within that pool, the first slide belonging to a `smoker` patient is used as the query — smoking status is the query selector, not an additional filter.

This is the correct pattern for clinical retrieval: first narrow the candidate pool to a clinically coherent subgroup using `MULTI_FILTERS`, then find the most morphologically similar slides within that subgroup. Running KNN against the full 1442-slide pool would return neighbors from unrelated sites and HPV statuses, diluting the clinical relevance of the results.

**What the output tells you:**

- The candidate pool size printed confirms `MULTI_FILTERS` was applied before KNN
- Similarity scores above the p95 baseline (0.659 from `embedding-quality-diagnostics`) are genuinely above-average morphological matches
- The UMAP plot shows the query (red star) and neighbors (orange diamonds) within the `MULTI_FILTERS` subgroup, colored by `hpv_association_p16` — consistent with the filter used

> UMAP proximity ≠ cosine similarity. Neighbors that appear far in 2D but score > 0.70 are still strong matches in 1536-d space.


In [20]:
# knn-query-smoking-status

SMOKING = "smoker"
K = 15

# Restrict KNN candidates to MULTI_FILTERS pool (Oropharynx + HPV-positive)
filtered_meta = apply_metadata_filters(metadata_df, MULTI_FILTERS, FILTERABLE_COLUMNS)
filtered_names = set(filtered_meta["slide_name"])
candidate_embeddings = {n: v for n, v in embeddings.items() if n in filtered_names}

matching = metadata_df[
    (metadata_df["smoking_status"] == SMOKING) &
    (metadata_df["slide_name"].isin(filtered_names))
]["slide_name"].tolist()

if not matching:
    print(f"No slides found for smoking_status = {SMOKING} within MULTI_FILTERS pool")
else:
    query_slide = matching[0]
    knn_results = find_k_nearest(
        embeddings, query_slide, k=K,
        candidate_embeddings=candidate_embeddings,
    )
    neighbor_names = [name for name, score in knn_results]
    print(f"Query: {query_slide} (smoking_status={SMOKING})")
    print(f"Candidate pool: {len(candidate_embeddings)} slides (MULTI_FILTERS applied)")
    print()
    knn_df = pd.DataFrame(knn_results, columns=["slide_name", "similarity"])
    knn_df.index = range(1, len(knn_df) + 1)
    knn_df.index.name = "rank"
    knn_df["similarity"] = knn_df["similarity"].round(4)
    _slide_meta = metadata_df.set_index("slide_name")[["age_at_initial_diagnosis", "year_of_initial_diagnosis"]]
    knn_df = knn_df.join(_slide_meta, on="slide_name")
    from IPython.display import display
    display(knn_df.style.background_gradient(subset=["similarity"], cmap="Blues"))
    fig_smoke = plot_umap_interactive(
        umap_df, metadata_df,
        color_by="hpv_association_p16",
        query_slide=query_slide,
        neighbors=neighbor_names,
        metadata_filters=MULTI_FILTERS,
    )
    fig_smoke.show()



Query: TumorCenter_HE_block10_x1_y1_patient246 (smoking_status=smoker)
Candidate pool: 626 slides (MULTI_FILTERS applied)



,slide_name,similarity,age_at_initial_diagnosis,year_of_initial_diagnosis
rank,,,,
1,TumorCenter_HE_block8_x5_y4_patient251,0.751500,57,2013
2,TumorCenter_HE_block9_x2_y10_patient372,0.740500,56,2014
3,TumorCenter_HE_block9_x3_y5_patient602,0.734400,59,2014
4,TumorCenter_HE_block1_x2_y6_patient564,0.727800,68,2016
5,TumorCenter_HE_block9_x3_y2_patient585,0.720500,69,2013
6,TumorCenter_HE_block5_x1_y4_patient511,0.717700,67,2018
7,TumorCenter_HE_block1_x1_y7_patient104,0.711200,55,2016
8,TumorCenter_HE_block10_x4_y1_patient470,0.704800,50,2012
9,TumorCenter_HE_block10_x6_y4_patient513,0.703300,64,2013


626 slides remaining after filtering


### Query by Histologic Type and Patient ID

Demonstrates targeted slide-level KNN: `MULTI_FILTERS` (Oropharynx + HPV positive or negative, 626 slides) restricts the candidate pool, then a specific patient and histologic type select the query slide within that pool.

**How the query is constructed:**
- `MULTI_FILTERS` builds the candidate pool — KNN only searches within these 626 slides
- `HISTOLOGIC_TYPE` selects slides of a specific histology as query candidates
- `PATIENT_ID` narrows to a specific patient — set to `None` to use the first matching slide across all patients
- `deduplicate_by_patient=True` — only the highest-scoring slide per patient is kept, giving a clean one-result-per-patient ranking

**Difference from `knn-query-smoking-status`:** both cells now use `MULTI_FILTERS` as the candidate pool. The difference is the query selector — smoking status there, histologic type + patient ID here. This cell also adds per-patient deduplication, making it closer to the patient-level matching in `match-table`.

**What the output tells you:**
- `Candidate pool` line confirms `MULTI_FILTERS` was applied
- Scores above the p95 baseline (0.659) are genuine morphological matches; scores near or below it are within population noise
- The UMAP plot shows the query (red star) and neighbors (orange diamonds) within the `MULTI_FILTERS` subgroup, colored by `hpv_association_p16` — compare with `knn-query-smoking-status` to see how a different query patient shifts the neighbor positions
- `histologic_type` has a silhouette of −0.082 — neighbors are not guaranteed to share the same histology; use `match-table` for the full composite score with metadata match

> If `PATIENT_ID` is not found for the given `HISTOLOGIC_TYPE`, a yellow warning box lists valid patient IDs — change `PATIENT_ID` to one of those values and re-run.


In [21]:
# knn-query-histologic-patient

PATIENT_ID = "015"
HISTOLOGIC_TYPE = "SCC_Conventional-Keratinizing"
K = 15

# Restrict KNN candidates to MULTI_FILTERS pool (Oropharynx + HPV positive or negative)
filtered_meta = apply_metadata_filters(metadata_df, MULTI_FILTERS, FILTERABLE_COLUMNS)
filtered_names = set(filtered_meta["slide_name"])
candidate_embeddings = {n: v for n, v in embeddings.items() if n in filtered_names}

# Filter by histologic type within the candidate pool
mask = metadata_df["histologic_type"] == HISTOLOGIC_TYPE
# Optionally narrow down to a specific patient
if PATIENT_ID is not None:
    mask = mask & metadata_df["slide_name"].str.contains(PATIENT_ID, na=False)

matching = metadata_df[mask]["slide_name"].tolist()
filter_desc = f"histologic_type={HISTOLOGIC_TYPE}" + (f", patient={PATIENT_ID}" if PATIENT_ID else "")

if not matching:
    _valid_pids = sorted({
        re.search(r'patient(\d+)', n).group(1)
        for n in metadata_df[metadata_df["histologic_type"] == HISTOLOGIC_TYPE]["slide_name"]
        if re.search(r'patient(\d+)', n)
    })
    from IPython.display import display, HTML
    display(HTML(f"""
        <div style='background:#fff3cd;border-left:6px solid #ffc107;padding:12px 16px;border-radius:4px;font-family:monospace'>
        <b>⚠️ No slides found for:</b> {filter_desc}<br><br>
        <b>Valid patient IDs for histologic_type={HISTOLOGIC_TYPE}:</b><br>
        <code>{_valid_pids[:10]}</code>
        </div>
    """))
else:
    query_slide = matching[0]
    knn_results = find_k_nearest(
        embeddings, query_slide, k=K,
        candidate_embeddings=candidate_embeddings,
        deduplicate_by_patient=True,
    )
    neighbor_names = [name for name, score in knn_results]
    print(f"Query: {query_slide} ({filter_desc})")
    print(f"Candidate pool: {len(candidate_embeddings)} slides (MULTI_FILTERS applied)")
    print(f"Matched {len(matching)} slide(s) for filter criteria")
    print()
    knn_df = pd.DataFrame(knn_results, columns=["slide_name", "similarity"])
    knn_df.index = range(1, len(knn_df) + 1)
    knn_df.index.name = "rank"
    knn_df["similarity"] = knn_df["similarity"].round(4)
    _slide_meta = metadata_df.set_index("slide_name")[["age_at_initial_diagnosis", "year_of_initial_diagnosis"]]
    knn_df = knn_df.join(_slide_meta, on="slide_name")
    from IPython.display import display
    display(knn_df.style.background_gradient(subset=["similarity"], cmap="Blues"))
    fig_hist = plot_umap_interactive(
        umap_df, metadata_df,
        color_by="hpv_association_p16",
        query_slide=query_slide,
        neighbors=neighbor_names,
        metadata_filters=MULTI_FILTERS,
    )
    fig_hist.show()



Query: TumorCenter_HE_block11_x3_y1_patient015 (histologic_type=SCC_Conventional-Keratinizing, patient=015)
Candidate pool: 626 slides (MULTI_FILTERS applied)
Matched 2 slide(s) for filter criteria



,slide_name,similarity,age_at_initial_diagnosis,year_of_initial_diagnosis
rank,,,,
1,TumorCenter_HE_block10_x5_y6_patient349,0.744600,58,2013
2,TumorCenter_HE_block5_x2_y4_patient511,0.719500,67,2018
3,TumorCenter_HE_block4_x4_y2_patient287,0.717600,64,2016
4,TumorCenter_HE_block6_x3_y8_patient405,0.707700,57,2018
5,TumorCenter_HE_block6_x4_y5_patient091,0.694400,69,2018
6,TumorCenter_HE_block2_x6_y4_patient359,0.691200,67,2015
7,TumorCenter_HE_block8_x2_y5_patient251,0.680300,57,2013
8,TumorCenter_HE_block10_x3_y11_patient119,0.675400,57,2012
9,TumorCenter_HE_block12_x5_y1_patient506,0.657500,48,2011


626 slides remaining after filtering


---
## What we've learned so far

The exploratory cells above established the following:

| Finding | Implication |
|---|---|
| All silhouette scores near zero or negative | H-optimus-0 slide-level embeddings don't cleanly separate clinical labels — retrieval is morphological proximity, not label matching. ABMIL aggregator is randomly initialised, producing a random weighted average empirically highly correlated with mean pooling but not equivalent to it. |
| p95 baseline = 0.659 | Only neighbors scoring above this are genuinely above-average matches |
| `hpv_association_p16` is the best-signal label (+0.015) | Filtering to Oropharynx + known HPV status gives the most coherent candidate pool |
| KNN neighbors score 0.67–0.82 within `MULTI_FILTERS` pool | Real morphological signal exists — the continuous similarity space is useful even without discrete clusters |
| UMAP shows heavy intermixing by site and histology | Visual proximity in 2D does not reliably predict cosine similarity in 1536-d |

**What slide-level KNN cannot do on its own:**
- Guarantee same histologic type or tumor site in results (silhouettes too weak)
- Account for tissue composition heterogeneity within a slide (mean-pooling loses this)
- Incorporate clinical metadata similarity into the ranking

**What the composite match table adds:**
The next section combines three signals into a single patient-level ranking — slide-level morphology (α=0.4), patch-level tissue composition profiles (β=0.4), and clinical metadata match (γ=0.2). This is the authoritative output for clinical trial matching.


## Composite match table for a query patient

### Step 1 — Determine morphological cluster count (composition K-sweep)

Before building composition profiles, we need to decide how many morphological clusters (K) to use for the patch-level KMeans codebook. Too few clusters merge distinct patch patterns; too many overfit to noise.

This cell samples patch embeddings from 5 patients directly from the H5 files referenced in the `h5file` column. These H5 files are stored in S3 (`s3://hancock-vol01-s3-.../HOptimus0/Hancock_WSI_PrimaryTumor_H5/`) and are streamed directly using `s3fs` — no local download required. Each H5 file contains the raw H-optimus-0 patch embeddings for one slide: one 1536-d vector per 224×224 patch, plus patch coordinates.

This is the same model that produced the slide-level embeddings in the CSV, but here we work with the individual patch vectors before any pooling. A typical slide contains thousands of patches, so the H5 files are large — streaming from S3 rather than downloading is what makes this practical at cohort scale.

> **Normalisation**: `sample_patch_features` L2-normalises every patch vector before returning them (see `composition.py: _load_features_parallel`). The K-sweep therefore operates on unit-norm vectors, and silhouette scores are computed with `metric="cosine"`. KMeans on unit-norm vectors is valid for cosine clustering because on the unit sphere `‖u−v‖² = 2 − 2·cos(u,v)`, so minimizing Euclidean distance is equivalent to maximizing cosine similarity. The same normalisation is applied inside `build_composition_profiles` when fitting the full codebook, so the K selected here transfers correctly. Note that L2-normalising before computing cosine similarity has no effect on the scores — cosine similarity is scale-invariant, so `cosine(u, v) == cosine(u/‖u‖, v/‖v‖)` exactly. Do not compare these K-sweep silhouette scores to the patient-level scores in `embedding-quality-diagnostics`: they measure different things — patch-level morphological cluster separation across thousands of patches vs patient-level label separation across ~720 mean-pooled slide vectors.

The K-sweep runs KMeans for K=3..15 on these L2-normalised patch vectors and picks the K with the highest silhouette score. The recommended K is then used in `load-or-build-profiles` to fit a shared codebook across all slides.

Working at patch level rather than slide level is what makes composition profiles complementary to the slide-level KNN signal — mean-pooling collapses all patch vectors into a single direction, discarding the proportional breakdown of morphological clusters. Two slides with different patch-pattern distributions may have similar mean-pooled embeddings if their patch distributions overlap in embedding space, yet their composition profiles will differ substantially.


In [22]:
# composition-k-sweep

# feats are L2-normalised by sample_patch_features (see composition.py: _load_features_parallel).
feats = sample_patch_features(metadata_df, n_patients=5, local_dir="../data/h5_cache")

print(f"K-sweep on {len(feats)} L2-normalised patches:")
results = {}
for k in range(3, 16):
    labels = MiniBatchKMeans(n_clusters=k, random_state=42, n_init="auto").fit_predict(feats)
    sil = silhouette_score(feats, labels, metric="cosine", sample_size=min(5000, len(feats)))
    results[k] = sil

best_k = max(results, key=results.get)
for k, sil in results.items():
    marker = " <-- best" if k == best_k else ""
    print(f"  K={k}: silhouette={sil:.4f}{marker}")

PROFILES_DIR = "../data/profiles"

try:
    patient_profiles, codebook = load_profiles(PROFILES_DIR)
except FileNotFoundError:
    print("No cached profiles — building from S3 (one-time)...")
    slide_profiles, codebook = build_composition_profiles_from_metadata(
        metadata_df,
        n_tissue_types=best_k,
        local_dir="../data/h5_cache",
        max_workers=16,
    )
    patient_profiles = build_patient_profiles(slide_profiles)
    save_profiles(patient_profiles, codebook, PROFILES_DIR)

pids = sorted(patient_profiles.keys())

print(f"\nRecommended n_tissue_types={best_k}")



Loading patch features from 5 local H5 files (5 patients requested) for K-sweep...


Loading H5 files:   0%|          | 0/5 [00:00<?, ?file/s]

K-sweep on 949 L2-normalised patches:
  K=3: silhouette=0.2441
  K=4: silhouette=0.3270
  K=5: silhouette=0.3545
  K=6: silhouette=0.3965 <-- best
  K=7: silhouette=0.3740
  K=8: silhouette=0.3531
  K=9: silhouette=0.3525
  K=10: silhouette=0.3278
  K=11: silhouette=0.3379
  K=12: silhouette=0.3250
  K=13: silhouette=0.3212
  K=14: silhouette=0.2666
  K=15: silhouette=0.2397
Loaded 720 patient profiles from ../data/profiles/patient_profiles.npz
Loaded codebook (K=6) from ../data/profiles/codebook.joblib

Recommended n_tissue_types=6


### Step 2 — Validate composition profiles (composition UMAP)

Projects patient-level composition profiles to 2D using UMAP to visually validate that the profiles capture meaningful structure. If composition profiles are working well, patients with similar morphological cluster distributions should cluster together regardless of clinical label.

This is a sanity check before running the match table — if the composition UMAP shows no structure at all, the β=0.4 composition weight in the composite score is adding noise rather than signal. Compare this plot with the slide-level UMAP in `umap-computation`: any additional grouping visible here reflects tissue composition patterns not captured by ABMIL-aggregated embeddings.


In [23]:
# composition-umap

# One row per patient with representative metadata
meta_patients = (
    metadata_df.copy()
)
meta_patients["slide_name"] = meta_patients["slide_name"].str.extract(r'(patient\d+)')
meta_patients = meta_patients.drop_duplicates("slide_name").reset_index(drop=True)

_pid_to_slide = {}
for sn in metadata_df["slide_name"]:
    import re as _re
    m = re.search(r'(patient\d+)', sn)
    if m:
        raw_pid = re.search(r'patient(\d+)', sn).group(1)
        _pid_to_slide.setdefault(raw_pid, m.group(1))

# pids = sorted(patient_profiles.keys())
comp_embeddings = {
    _pid_to_slide[pid]: patient_profiles[pid].astype(float)
    for pid in pids
    if pid in _pid_to_slide
}
comp_umap_df = run_umap(comp_embeddings)

fig = plot_umap_interactive(
    comp_umap_df,
    meta_patients,
    color_by="histologic_type",
)
fig.show()



/Users/ganttmeredith/Desktop/Kiro_BioFM/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


### Step 3 — Morphological match table

Runs the composite patient ranking for the query patient defined in `knn-query-histologic-patient`. See the `## 9. Morphological Match Table` section below for full documentation.


### Morphological Match Table

Ranks all patients by composite similarity to the query patient. Each row shows:
- `composite` — weighted score: α×slide_sim + β×comp_sim + γ×meta_sim (default α=0.4, β=0.4, γ=0.2)
- `slide_sim` — max pairwise cosine similarity across all (slide_a, slide_b) combinations, computed directly on the per-slide ABMIL-aggregated 1536-d vectors (no second mean-pool). Max-aggregation has a known positive bias for patients with more slides; in this dataset 753/760 patients have exactly 2 slides (stdev=0.19) so the effect is negligible.
- `comp_sim` — cosine similarity of unsupervised morphological cluster proportion profiles (patch-level; clusters are not validated against histological ground truth)
- `meta_sim` — fraction of exactly-matching fields across: `primary_tumor_site`, `histologic_type`, `hpv_association_p16`, `pT_stage`, `pN_stage` (fields with unknown/nan values are excluded from the denominator). **Known limitation**: staging fields use exact string matching — adjacent stages (pT1 vs pT2) and substage variants (pT1 vs pT1a) both score 0, identical to distant stages (pT1 vs pT4b). Ordinal partial credit is not yet implemented.
- `slide_sim_vs_baseline` — slide_sim minus the population mean from the null distribution (requires `embedding-quality-diagnostics` to have been run; baseline p95 varies ±0.01 across random seeds — re-run that cell if comparing results across sessions)

> ⚠️ **Dependency**: `slide_sim_vs_baseline` uses `null_dist` computed in `embedding-quality-diagnostics`. Run cells in order, or re-run `embedding-quality-diagnostics` before interpreting this column.

This is the primary output for clinical trial matching. Rank by `composite` for the full signal,
or by `slide_sim` alone to prioritise pure morphological similarity.

In [24]:
# match-table

# patient_profiles keys match what re.search(r'patient(\d+)') extracts from slide names.
# PATIENT_ID must use that same form (e.g. '727', '038', '492').
_embed_pids = {re.search(r'patient(\d+)', n).group(1) for n in embeddings if re.search(r'patient(\d+)', n)}
if PATIENT_ID not in patient_profiles:
    # Derive suggestions from the same histologic type filter used in the KNN cell above
    _mask = metadata_df["histologic_type"] == HISTOLOGIC_TYPE
    _filtered_pids = {
        re.search(r'patient(\d+)', n).group(1)
        for n in metadata_df[_mask]["slide_name"]
        if re.search(r'patient(\d+)', n)
    }
    _valid = sorted(p for p in _filtered_pids if p in patient_profiles and p in _embed_pids)
    from IPython.display import display, HTML
    display(HTML(f"""
        <div style='background:#fff3cd;border-left:6px solid #ffc107;padding:12px 16px;border-radius:4px;font-family:monospace'>
        <b>⚠️ No slides found for:</b> PATIENT_ID={PATIENT_ID} with histologic_type={HISTOLOGIC_TYPE}<br><br>
        <b>Valid patient IDs for histologic_type={HISTOLOGIC_TYPE}:</b><br>
        <code>{_valid[:10]}</code>
        </div>
    """))
else:
    _valid_candidates = [p for p in patient_profiles if p in _embed_pids and p != PATIENT_ID]
    print(f"Query: patient {PATIENT_ID} | Candidates with embeddings: {len(_valid_candidates)} / {len(patient_profiles)}")

    results = rank_patients_by_composite(
        PATIENT_ID,
        _valid_candidates,
        embeddings,
        patient_profiles,
        metadata_df,
        alpha=0.4,  # slide-level cosine
        beta=0.4,   # composition profile
        gamma=0.2,  # metadata match
        slide_aggregation="max",  # max pairwise across slides — avoids double mean-pool
        display_columns=[
            "primary_tumor_site",
            "histologic_type",
            "hpv_association_p16",
            "pT_stage",
            "pN_stage",       # included in meta_sim score — shown here for transparency
            "smoking_status",
            "age_at_initial_diagnosis",
            "year_of_initial_diagnosis",
        ],
    )
    # Add slide_sim vs population baseline.
    # Requires null_dist from `embedding-quality-diagnostics` — re-run that cell if baseline looks wrong.
    # Baseline p95 varies ~±0.01 across random seeds (observed range 0.658–0.680 over 20 seeds).
    results["slide_sim_vs_baseline"] = (results["slide_sim"] - null_dist["mean"]).round(4)
    cols = list(results.columns)
    sim_idx = cols.index("slide_sim")
    cols.insert(sim_idx + 1, cols.pop(cols.index("slide_sim_vs_baseline")))
    results = results[cols]
    display(results.head(16))



Query: patient 015 | Candidates with embeddings: 719 / 720


,rank,patient_id,composite,slide_sim,slide_sim_vs_baseline,comp_sim,meta_sim,primary_tumor_site,histologic_type,hpv_association_p16,pT_stage,pN_stage,smoking_status,age_at_initial_diagnosis,year_of_initial_diagnosis
0,0,015 (query),1.0000,1.0000,0.5633,1.0000,1.0,Oropharynx,SCC_Conventional-Keratinizing,positive,pT1,pN0,former,54,2012
1,1,511,0.8248,0.7701,0.3334,0.9918,0.6,Oropharynx,SCC_Conventional-Keratinizing,negative,pT1,pN3b,smoker,67,2018
2,2,506,0.8190,0.6960,0.2593,0.9515,0.8,Oropharynx,SCC_Conventional-Keratinizing,negative,pT1,pN0,former,48,2011
3,3,419,0.8152,0.6390,0.2023,0.9991,0.8,Oropharynx,SCC_Conventional-Keratinizing,positive,pT1,pN3,smoker,61,2013
4,4,267,0.7855,0.7648,0.3281,0.9991,0.4,Larynx,SCC_Conventional-Keratinizing,not_tested,pT3,pN0,former,70,2014
5,5,279,0.7676,0.6294,0.1927,0.9895,0.6,Oral_Cavity,SCC_Conventional-Keratinizing,not_tested,pT1,pN0,former,49,2015
6,6,562,0.7661,0.6408,0.2041,0.8743,0.8,Oropharynx,SCC_Conventional-Keratinizing,negative,pT1,pN0,former,55,2019
7,7,558,0.7653,0.5683,0.1316,0.9450,0.8,Oropharynx,SCC_Conventional-Keratinizing,negative,pT1,pN0,smoker,66,2012
8,8,091,0.7591,0.6944,0.2577,0.9033,0.6,Oropharynx,SCC_Conventional-Keratinizing,negative,pT1,pN1,smoker,69,2018
9,9,359,0.7562,0.6912,0.2545,0.9993,0.4,Oropharynx,SCC_Conventional-Keratinizing,negative,pT3,pN3,smoker,67,2015


---
## Glossary

### Clinical & pathology terms

| Term | Definition |
|---|---|
| **HNSCC** | Head and Neck Squamous Cell Carcinoma. A group of cancers arising from the mucosal surfaces of the head and neck — including the oropharynx, oral cavity, larynx, and hypopharynx. All subtypes in this dataset are squamous carcinomas, which is why H&E morphology overlaps heavily across sites. |
| **H&E** | Haematoxylin and Eosin stain. The standard histological stain used in pathology. Haematoxylin stains nuclei blue/purple; eosin stains cytoplasm and extracellular matrix pink. All slides in this dataset are H&E-stained. |
| **WSI** | Whole Slide Image. A high-resolution digital scan of an entire histology glass slide, typically at 20× or 40× magnification. WSIs are too large to process as a single image and are divided into patches. |
| **TMA** | Tissue Microarray. A technique where small cores from many patient tissue blocks are arrayed on a single slide, enabling high-throughput staining and imaging. The `tma_tumorcenter_HE.csv` source file maps TMA grid coordinates to patient IDs. |
| **SCC** | Squamous Cell Carcinoma. A cancer arising from squamous epithelial cells. Subtypes in this dataset include `SCC_Conventional-Keratinizing`, `SCC_Basaloid`, `SCC_Sarcomatoid`, `SCC_Conventional-NonKeratinizing`, `SCC_Lymphoepithelial`, and `SCC_Acantholytic`. |
| **HPV** | Human Papillomavirus. A viral infection strongly associated with oropharyngeal SCC. HPV-positive tumours (`hpv_association_p16 = positive`) have distinct morphology, better prognosis, and different treatment response compared to HPV-negative tumours. |
| **p16** | A surrogate immunohistochemical marker for HPV infection. Overexpression of p16 (CDKN2A) is used as a proxy for high-risk HPV status in oropharyngeal cancers. |
| **pT stage** | Pathological tumour stage (TNM classification). Describes the size and local extent of the primary tumour after surgical resection. Ranges from pT1 (small, localised) to pT4 (large or invading adjacent structures). |
| **pN stage** | Pathological nodal stage. Describes the extent of regional lymph node involvement. pN0 = no nodal metastasis; pN2b = multiple ipsilateral nodes; etc. |
| **Perineural invasion (Pn)** | Tumour cells invading the space surrounding nerves. A pathological feature associated with higher recurrence risk. |
| **Lymphovascular invasion** | Tumour cells invading lymphatic or blood vessels. Associated with increased metastatic potential. |
| **Perinodal invasion** | Tumour extension beyond the capsule of a lymph node into surrounding tissue. Also called extranodal extension (ENE). |

### Machine learning & embedding terms

| Term | Definition |
|---|---|
| **Embedding** | A dense numerical vector representation of an image (or other data) learned by a neural network. In this pipeline, each slide is represented by a 1536-dimensional embedding produced by H-optimus-0. |
| **H-optimus-0** | A 1.1B parameter Vision Transformer (ViT) foundation model for histopathology, developed by Bioptimus. Pretrained on a large proprietary collection of H&E WSIs. Produces 1536-d patch embeddings from 224×224 images. |
| **ViT** | Vision Transformer. A neural network architecture that applies the transformer self-attention mechanism to image patches, treating each patch as a token. H-optimus-0 is a ViT. |
| **Mean pooling** | Averaging all patch embeddings for a slide into a single vector. Simple and parameter-free, but loses spatial and heterogeneity information. In this pipeline, slide-level embeddings are produced by ABMIL aggregation rather than plain mean pooling — with randomly initialised weights the aggregator produces a random weighted average that is empirically highly correlated with mean pooling, but the two are not equivalent. The distinction only becomes meaningful once the aggregator is trained on a task-specific objective. |
| **KNN** | K-Nearest Neighbours. A retrieval method that finds the k most similar items to a query by distance in embedding space. This pipeline uses cosine distance in the original 1536-d space. |
| **Cosine similarity** | A measure of similarity between two vectors based on the angle between them, ranging from −1 (opposite) to +1 (identical direction). Used throughout this pipeline for slide and composition profile comparison. |
| **Silhouette score** | A metric measuring how well-separated clusters are in an embedding space. Ranges from −1 to +1. Near zero means no cluster structure; negative means worse than random. In high-dimensional cosine spaces scores are typically compressed toward zero — the task-specific interpretation table in `embedding-quality-diagnostics` applies rather than the standard Kaufman & Rousseeuw thresholds. |
| **UMAP** | Uniform Manifold Approximation and Projection. A dimensionality reduction algorithm that projects high-dimensional embeddings to 2D or 3D for visualisation. Used here for exploration only — not for matching. |
| **Composition profile** | A fixed-length vector describing the proportion of each unsupervised morphological cluster in a slide or patient, derived from patch-level KMeans clustering of H5 embeddings. Captures morphological heterogeneity that mean-pooling destroys. Clusters are not validated against histological ground truth — they reflect recurring patterns in the patch embedding space. |
| **LoRA** | Low-Rank Adaptation. A parameter-efficient fine-tuning technique that inserts small trainable rank-decomposition matrices into transformer layers, updating <1% of model parameters while adapting the model to a new task. |

### Infrastructure & data terms

| Term | Definition |
|---|---|
| **H5 / HDF5** | Hierarchical Data Format version 5. A binary file format for storing large numerical arrays. Each slide's patch embeddings and coordinates are stored in an H5 file on S3, referenced by the `h5file` column. |
| **HNSW** | Hierarchical Navigable Small World. A graph-based approximate nearest-neighbour index that enables sub-linear KNN search. Used in production vector stores (OpenSearch `knn_vector`, pgvector, Pinecone) to replace brute-force cosine search over large embedding collections. |
| **RAG** | Retrieval-Augmented Generation. A pattern where a retrieval system (here, KNN over slide embeddings) fetches relevant context that is passed to a language model to ground its response. In this pipeline, retrieved patient profiles and metadata would form the context for a clinical LLM. |
| **OpenSearch** | An open-source distributed search and analytics engine. Used in production as the vector store for slide embeddings, with HNSW indexing via the `knn_vector` field type and metadata pre-filtering on clinical fields. |
| **S3** | Amazon Simple Storage Service. Object storage used to host the H5 patch embedding files and the slide embeddings CSV. |
| **SageMaker HyperPod** | An AWS service providing purpose-built, resilient infrastructure for large-scale distributed model training. Supports automatic node failure recovery, tensor and data parallelism, and training recipes for common model architectures. |
| **SMP** | SageMaker Model Parallel. An AWS library for distributing large model training across multiple GPUs and nodes using tensor parallelism, pipeline parallelism, or both. |
| **Hancock dataset** | The HNSCC cohort used in this notebook. A **monocentric** collection of TMA slides from head and neck cancer patients at the University Hospital Erlangen, with matched clinical, pathological, and blood laboratory data. |


---

## Summary

This notebook produced a ranked list of morphologically and clinically similar patients for a given query patient.

| Output | Description |
|---|---|
| UMAP plot | 2D visual overview of the embedding space — exploration only, not used for matching |
| Cluster cross-tabulation | Unsupervised K-means clusters cross-tabulated against clinical labels — reveals what structure the embeddings organise by |
| KNN results (slide-level) | Top-k most morphologically similar slides to the query in 1536-d cosine space |
| `results` match table | Patient-level composite ranking: α×slide_sim + β×comp_sim + γ×meta_sim |

**Interpreting the match table:**
- `composite` — the primary ranking signal; use this to shortlist candidates
- `slide_sim_vs_baseline` — how far above the population mean the morphological similarity is; values ≤ 0 are within noise
- `meta_sim` — fraction of exactly-matching clinical fields; use metadata filters to pre-restrict the candidate pool for higher-precision results

**Improving results:** the weak silhouette scores (~0) reflect an untrained ABMIL aggregator. Fine-tuning H-optimus-0 with LoRA on HNSCC labels (see *Improving signal* section above) is the highest-leverage next step — expected to move `histologic_type` and `primary_tumor_site` silhouettes from −0.08 to +0.10–0.20.

> **Demo vs production note**
>
> This notebook reads embeddings from a flat CSV for portability. In production, the embeddings would live in a **vector store** (e.g. OpenSearch, S3Vectors, pgvector, or Pinecone with HNSW indexing), which supports:
> - Sub-linear approximate nearest-neighbour search via HNSW — orders of magnitude faster than brute-force cosine over a full CSV at scale
> - Metadata filtering pushed down to the index, avoiding full scans
> - Incremental upserts as new slides are processed, without rewriting the full dataset
>
> The retrieval logic in `umap_retrieval/retrieval.py` is intentionally decoupled from the data source — swapping the CSV for a vector store requires only changing how `embeddings` is populated, not the KNN or composite scoring code.